# SARR ETL — Libraries.io (PyPI) → Embed (GPU) → Qdrant Cloud

Uses `google.cloud.bigquery.Client` to **read** public Libraries.io data
(`bigquery-public-data.libraries_io.projects` + `repositories`). No table creation.

1. Runtime → **GPU**
2. Set `GCP_PROJECT_ID` to **your** GCP project (billing/quota only)
3. First load: `LAST_UPDATE_DATE = "1970-01-01"`
4. Later: set watermark to previous max `latest_release_publish_timestamp`

In [ ]:
# 1) Runtime → Change runtime type → T4 GPU (or better)
# 2) Get the code into Colab (pick ONE)

# Option A — clone from GitHub (private repos need a token)
# !git clone https://github.com/YOUR_ORG/sarr-recommendation-api.git
# %cd sarr-recommendation-api

# Option B — upload the repo zip via Colab Files, then:
# !unzip -q sarr-recommendation-api.zip
# %cd sarr-recommendation-api

!pip install -q -e ".[etl]"

In [ ]:
import os
from pathlib import Path

# YOUR GCP project — billing/quota only
os.environ["GCP_PROJECT_ID"] = "your-gcp-project-id"

# Libraries.io public source (your working MVP query)
os.environ["BQ_SOURCE_PROJECT"] = "bigquery-public-data"
os.environ["BQ_DATASET"] = "libraries_io"
os.environ["BQ_TABLE"] = "projects"

os.environ["QDRANT_URL"] = "https://YOUR-CLUSTER.aws.cloud.qdrant.io"
os.environ["QDRANT_API_KEY"] = "YOUR_QDRANT_KEY"
os.environ["QDRANT_COLLECTION"] = "sarr_pypi"
os.environ["EMBEDDING_MODEL"] = "BAAI/bge-small-en-v1.5"
os.environ["EMBEDDING_DIM"] = "384"
os.environ["LAST_UPDATE_DATE"] = "1970-01-01"  # full load; change later for incremental

from google.colab import auth
auth.authenticate_user()

Path("data").mkdir(exist_ok=True)
print("Auth OK. Source: bigquery-public-data.libraries_io (PyPI)")
print("Billing project:", os.environ["GCP_PROJECT_ID"])

In [ ]:
# Smoke-test BigQuery client (optional — run before full ETL)
from google.cloud import bigquery
from sarr.common.config import get_settings

get_settings.cache_clear()
settings = get_settings()
client = bigquery.Client(project=settings.gcp_project_id)

smoke_sql = f"""
SELECT
  p.name,
  p.description,
  CAST(COALESCE(r.stars_count, 0) AS INT64) AS stars,
  p.latest_release_publish_timestamp
FROM `{settings.bq_source_project}.{settings.bq_dataset}.projects` AS p
LEFT JOIN `{settings.bq_source_project}.{settings.bq_dataset}.repositories` AS r
  ON p.repository_id = r.id
WHERE p.platform = 'Pypi'
  AND p.description IS NOT NULL
  AND TRIM(p.description) != ''
ORDER BY p.latest_release_publish_timestamp DESC
LIMIT 5
"""
print(smoke_sql)
for row in client.query(smoke_sql):
    print(dict(row))

In [ ]:
import torch
from sarr.common.config import get_settings
from sarr.etl.embed import BatchEmbedder
from sarr.etl.pipeline import run_etl

assert torch.cuda.is_available(), "Enable a GPU runtime before running ETL"

get_settings.cache_clear()
settings = get_settings()
print(
    "Source table:",
    f"{settings.bq_source_project}.{settings.bq_dataset}.{settings.bq_table}",
)
print("Billing project:", settings.gcp_project_id)
print("Qdrant:", settings.qdrant_url, "collection=", settings.qdrant_collection)
print("Embed model:", settings.embedding_model)

embedder = BatchEmbedder(settings.embedding_model, device="cuda")

stats = run_etl(
    last_update_date=settings.last_update_date,
    batch_size=64,  # lower to 32 if you hit GPU OOM
    watermark_path="data/last_update_date.txt",
    settings=settings,
    embedder=embedder,
)
print(stats)
print("New watermark saved to data/last_update_date.txt — reuse it next month")